In [2]:
import pandas as pd
import numpy as np
from causalexplain import GraphDiscovery
from graphviz import Digraph
from pathlib import Path
import re

def draw_graphviz_dag(adj, nodes,labels, out_path, engine="dot"):
    #guards to check for adjacency matrix dimension
    adj = np.asarray(adj)
    print("draw_graphviz_dag adj ndim:", adj.ndim, "shape:", adj.shape)

    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]
    nodes = list(nodes)[:n]
    
    # Restore original labels where possible
    labels = labels

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    g.attr(rankdir="TB")  # left-to-right; change to "TB" if you prefer top-down
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10"
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7"
    )

    # Add nodes with restored labels
    for clean_name, label in zip(nodes, labels):
        g.node(clean_name, label=label)

    # Add edges
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)



#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

#path to put results
output_dir = project_root/"results"/"graphs_ReX"
output_dir.mkdir(parents=True,exist_ok=True)

In [3]:
# Map from cleaned -> original for restoring labels in plots
def clean_and_encode_df(df: pd.DataFrame):
    """
    - Drop rows with NA.
    - Clean column names for algorithms (letters+digits, start with letter).
    - One-hot encode non-numeric columns.
    Returns:
      df_enc: encoded numeric DataFrame
      clean_to_orig: dict {clean_name: original_name}
    """
    df = df.dropna().copy()
    if df.empty:
        raise ValueError("Data frame is empty after dropna().")

    # 1) Clean base column names
    orig_cols = list(df.columns)
    clean_cols = []
    for c in orig_cols:
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)  # remove underscores, spaces, etc.
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        clean_cols.append(c2)
    df.columns = clean_cols
    clean_to_orig = dict(zip(clean_cols, orig_cols))

    # 2) One-hot encode non-numeric columns
    non_numeric = df.select_dtypes(exclude=["number"]).columns
    if len(non_numeric) > 0:
        df_enc = pd.get_dummies(df, columns=list(non_numeric), drop_first=False, dtype=float)
    else:
        df_enc = df.astype(float)

    return df_enc, clean_to_orig

In [4]:
def dot_to_adjacency(dot_path):
    G = nx.drawing.nx_pydot.read_dot(str(dot_path))
    nodes = list(G.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    p = len(nodes)
    adj = np.zeros((p, p), dtype=int)
    for u, v in G.edges():
        i = idx[u]
        j = idx[v]
        adj[i, j] = 1

    adj = np.asarray(adj)
    if adj.ndim == 1:              # safety
        adj = adj.reshape(1, 1)
    return adj, nodes

In [5]:
import networkx as nx
def run_rex(df, experiment_name="rex_exp"):
    #Graph Discovery expects a file path for dataframe
    tmp_csv = "tmp_rex_input.csv"
    df = clean_name(df)
    df.to_csv(tmp_csv, index=False) #write df to a csv temporarily

    #init graph discovery instance
    gd = GraphDiscovery(experiment_name=experiment_name, model_type="rex",csv_filename= tmp_csv)

    gd.run(quiet=True)

    dot_path = output_dir / f"{experiment_name}.dot"
    gd.export_dag(str(dot_path))

    # Read DOT with networkx and build adjacency
    adj, nodes = dot_to_adjacency(dot_path)
    print("ReX adj ndim:", adj.ndim, "shape:", adj.shape)

    return adj, nodes

In [7]:
def clean_name(df):
    #makes column names ReX friendly i.e start with letter, contain only letters and numbers
    new_cols = []
    for c in df.columns:
        # remove underscores and other non-alphanumeric
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)
        # if it doesn't start with a letter, prefix with 'X'
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        new_cols.append(c2)
    df2 = df.copy()
    df2.columns = new_cols
    return df2

In [1]:
#scoring utility function given two adjacency matricies
def score_graph(W_est, W_true, labels_est, labels_true):
    #number of variables
    p = W_est.shape[0]

    #node orderings
    labels_est = list(labels_est)
    labels_true = list(labels_true)
    common = sorted(set(labels_est) & set(labels_true))

    #map ids
    idx_est = [labels_est.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_est_aligned = np.asarray(W_est)[np.ix_(idx_est, idx_est)]
    W_true_aligned = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    est = (W_est!=0).astype(int)
    true = (W_true !=0).astype(int)

    #true positive, false positive, false negative, true negative
    tp = np.sum((est==1) & (true==1))
    fp = np.sum((est==1) & (true == 0))
    fn = np.sum((est==0) & (true == 1))
    tn = np.sum((est==0) & (true == 0))

    #structural hamming distance, true positive rate, false positive rate
    shd = fp + fn
    tpr = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    fpr = fp/(fp+tn) if (fp + tn)>0 else np.nan
    fdr = fp/(tp+fp) if (tp+fp)>0 else np.nan

    #store scores in dictionary format for each 'experiment'
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), SHD=int(shd), TPR = tpr, FPR = fpr, FDR= fdr)

In [8]:
csv_files = sorted(cp_root.rglob("*.csv"))
results = []

for csv_path in csv_files:
    rel = csv_path.relative_to(cp_root)

    #we only examine those with known ground truths i.e with _truth
    if csv_path.stem.endswith("_truth"):
        continue

    #expected ground truth path
    truth_path = csv_path.with_name(csv_path.stem + "_truth.csv")
    if not truth_path.exists():
        print("  Skipped (no ground-truth file)")
        continue
        
    print(f"Processing (ReX): {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        orig_labels = df.columns

        adj, nodes = run_rex(df, experiment_name=f"rex__{csv_path.stem}")
        print("  ReX returned adj shape:", getattr(adj, "shape", None),
              "nodes:", len(nodes))
        
        if adj is None or np.size(adj) == 0:
            print("  No adjacency matrix from ReX; skipping plot.")
            continue
            
        parts = rel.parts
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__ReX.png"
        out_path = output_dir / out_name

        draw_graphviz_dag(adj, nodes, orig_labels, out_path)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

        #load ground truth and score
        gt_df = pd.read_csv(truth_path)
        gt_df.index.name = None
        gt_df.columns.name = None

        W_true = gt_df.to_numpy()
            
        # compute scores
        metrics = score_graph(adj, W_true, labels_est=list(df.columns), labels_true=list(gt_df.columns))
        print("metrics computed")
        print(metrics)
        metrics.update(
            dict(
                scenario=scenario,
                dataset=name_no_ext,
                algo="rex"
            )
        )
        results.append(metrics)
        print(f"  SHD={metrics['SHD']}, TPR={metrics['TPR']:.3f}, FDR={metrics['FDR']:.3f}")


    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

scores_df=pd.DataFrame(results)

  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing (ReX): casual_effect/device_failure_data.csv
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
ReX adj ndim: 2 shape: (7, 7)
  ReX returned adj shape: (7, 7) nodes: 7
draw_graphviz_dag adj ndim: 2 shape: (7, 7)
  Saved graph to results/graphs_ReX/casual_effect__device_failure_data__ReX.png
metrics computed
{'TP': 1, 'FP': 10, 'FN': 8, 'TN': 30, 'SHD': 18, 'TPR': 0.1111111111111111, 'FPR': 0.25, 'FDR': 0.9090909090909091}
  SHD

In [9]:
scores_df

,TP,FP,FN,TN,SHD,TPR,FPR,FDR,scenario,dataset,algo
0,1,10,8,30,18,0.1111,0.2500,0.9091,casual_effect,device_failure_data,rex
1,3,1,6,39,7,0.3333,0.0250,0.2500,casual_effect,student_tutoring_data,rex
2,0,2,3,11,5,0.0000,0.1538,1.0000,causal_direction_iv,clinical_trial_sem,rex
3,0,0,3,13,3,0.0000,0.0000,NaN,causal_direction_iv,ecommerce_sem,rex
4,1,1,2,12,3,0.3333,0.0769,0.5000,causal_direction_iv,environment_sem,rex
5,0,3,3,10,6,0.0000,0.2308,1.0000,causal_direction_iv,marketing_sem,rex
6,0,2,4,10,6,0.0000,0.1667,1.0000,counterfactual_reasoning,climate_impact_sem,rex
7,1,1,3,11,4,0.2500,0.0833,0.5000,counterfactual_reasoning,clinical_trial_sem,rex
8,0,2,4,10,6,0.0000,0.1667,1.0000,counterfactual_reasoning,education_performance_sem,rex
9,0,2,4,10,6,0.0000,0.1667,1.0000,counterfactual_reasoning,investment_outcome_sem,rex


In [10]:
scores_path = project_root / "results" / "scores"/"scores_rex.csv"
scores_path.parent.mkdir(parents=True, exist_ok=True)
scores_df.to_csv(scores_path, index=False)